# 12 · Klassen und Objekte

**Lernziele**

- Du definierst Klassen mit Attributen und Methoden.
- Du erzeugst Objekte und arbeitest mit ihnen.
- Du gibst Objekte lesbar aus und vergleichst sie.
- Du nutzt `@dataclass` für reine Datenklassen.
- Du legst feste Auswahlwerte als `Enum` an.
- Du erweiterst Klassen durch Vererbung und überschreibst Methoden.

> Grundlage: Inden, *Einstieg in Python*, Kapitel 4 und 6.

## Warum Klassen?

Bisher hast du Daten (z. B. Dictionaries) und Funktionen getrennt. Eine **Klasse** bündelt beides: Sie beschreibt, welche Daten ein Ding hat (**Attribute**) und was es kann (**Methoden**). Ein **Objekt** ist ein konkretes Exemplar einer Klasse.

| Klasse (Bauplan) | Objekte (Exemplare) |
| --- | --- |
| `Car` | Toms Audi, Jims VW |
| `BankAccount` | Konto 4711, Konto 0815 |

## Eine erste Klasse

`__init__` wird beim Erzeugen aufgerufen und legt die Attribute an. `self` ist das Objekt selbst.

In [ ]:
class Car:
    def __init__(self, brand, color, horse_power):
        self.brand = brand
        self.color = color
        self.horse_power = horse_power

    def paint(self, new_color):
        self.color = new_color

    def apply_tuning_kit(self):
        self.horse_power += 150


my_car = Car("VW", "orange", 123)
print(my_car.brand, my_car.color, my_car.horse_power)

my_car.paint("green")
my_car.apply_tuning_kit()
print(my_car.color, my_car.horse_power)

> Für Java- und C#-Umsteiger: Es gibt kein `new`, kein `this` (dafür explizit `self`), und Attribute werden nicht vorab deklariert.

## Lesbare Ausgabe mit `__str__`

Ohne Hilfe sieht ein Objekt in der Ausgabe kryptisch aus:

In [ ]:
print(my_car)

Methoden mit doppelten Unterstrichen (**Dunder-Methoden**) haben eine besondere Bedeutung. `__str__` legt fest, was `print()` zeigt:

In [ ]:
class Car:
    def __init__(self, brand, color, horse_power):
        self.brand = brand
        self.color = color
        self.horse_power = horse_power

    def __str__(self):
        return f"{self.brand} ({self.color}, {self.horse_power} PS)"


print(Car("Audi", "blue", 275))

## Objekte vergleichen

`==` vergleicht standardmäßig, ob es **dasselbe** Objekt ist, nicht ob die Inhalte gleich sind:

In [ ]:
toms_car = Car("Audi", "blue", 275)
jims_car = Car("Audi", "blue", 275)
print(toms_car == jims_car)

Man könnte jetzt `__eq__` selbst schreiben. Einfacher geht es mit einer **Datenklasse**.

## Datenklassen mit `@dataclass`

Für Klassen, die vor allem Daten halten, erzeugt `@dataclass` automatisch `__init__`, eine lesbare Ausgabe und `==`. Die Attribute werden mit Type Hints angegeben.

In [ ]:
from dataclasses import dataclass


@dataclass
class Car:
    brand: str
    color: str
    horse_power: int

    def apply_tuning_kit(self) -> None:
        self.horse_power += 150


toms_car = Car("Audi", "blue", 275)
jims_car = Car("Audi", "blue", 275)
print(toms_car)
print(toms_car == jims_car)

> Faustregel: Für reine Daten `@dataclass`, für Objekte mit viel Verhalten eine normale Klasse. Beides sind ganz normale Klassen.

## Interne Attribute

Python kennt kein `private`. Die **Konvention**: Ein Unterstrich am Anfang bedeutet „intern, bitte nicht von außen ändern“. Der Zugriff erfolgt über Methoden.

In [ ]:
class Counter:
    def __init__(self):
        self._value = 0

    @property
    def value(self) -> int:
        return self._value

    def increment(self) -> None:
        self._value += 1

    def reset(self) -> None:
        self._value = 0


counter = Counter()
counter.increment()
counter.increment()
print(counter.value)
counter.reset()
print(counter.value)

`@property` macht `value` von außen **lesbar wie ein Attribut**, aber nicht schreibbar:

In [ ]:
counter.value = 100

## Aufzählungen mit `Enum`

Manche Werte haben nur wenige erlaubte Möglichkeiten: Ampelfarben, Spielkartenfarben, der Status einer Bestellung. Als Strings wie `"rot"` sind sie anfällig für Tippfehler: `"Rot"` oder `"rto"` fällt erst spät oder gar nicht auf. Eine **Aufzählung** (*Enumeration*) legt die erlaubten Werte als Klasse fest. Die Mitglieder schreibt man in Großbuchstaben, weil es Konstanten sind:

In [ ]:
from enum import Enum, StrEnum, auto


class TrafficLight(Enum):
    RED = 1
    YELLOW = 2
    GREEN = 3


light = TrafficLight.RED
print(light)
print(light.name, light.value)
print(light == TrafficLight.RED)

Jedes Mitglied hat einen Namen (`name`) und einen Wert (`value`). Verglichen wird mit `==` oder `is`. Ein Tippfehler fällt sofort auf, statt still falsch zu laufen:

In [ ]:
TrafficLight.RDE

Über eine Enum kannst du iterieren, in der Reihenfolge der Definition:

In [ ]:
for color in TrafficLight:
    print(color.name, color.value)

print(TrafficLight(2))  # aus einem gespeicherten Wert wird wieder ein Mitglied

Sind dir die konkreten Werte egal, vergibt `auto()` sie. `StrEnum` verhält sich zusätzlich wie ein String und bekommt mit `auto()` den Namen in Kleinbuchstaben als Wert. Das ist praktisch, wenn die Werte in Dateien, JSON oder Ausgaben landen:

In [ ]:
class OrderStatus(StrEnum):
    OPEN = auto()  # Wert: "open"
    SHIPPED = auto()
    DELIVERED = auto()


status = OrderStatus.SHIPPED
print(status, status == "shipped")
print(f"Status: {status}")

Enums und `match` (Notebook 03) passen gut zusammen. Weil `TrafficLight.RED` einen Punkt enthält, **vergleicht** `case` mit diesem Wert, statt ihn einzufangen:

In [ ]:
def next_light(light: TrafficLight) -> TrafficLight:
    match light:  # vereinfacht, ohne Rot-Gelb
        case TrafficLight.RED:
            return TrafficLight.GREEN
        case TrafficLight.GREEN:
            return TrafficLight.YELLOW
        case TrafficLight.YELLOW:
            return TrafficLight.RED


light = TrafficLight.RED
for _ in range(4):
    print(light.name)
    light = next_light(light)

## Vererbung

Eine Klasse kann von einer anderen **erben**: Sie übernimmt alles und kann Methoden ergänzen oder **überschreiben**. Mit `super()` rufst du die Version der Elternklasse auf.

Beispiel: Ein Punktezähler, der bei 100 überläuft und dafür ein Bonusleben vergibt.

In [ ]:
class CounterWithOverflow(Counter):
    MAX_VALUE = 100

    def __init__(self):
        super().__init__()
        self.overflow_count = 0

    def increment(self) -> None:
        super().increment()
        if self.value >= self.MAX_VALUE:
            self._value = 0
            self.overflow_count += 1


points = CounterWithOverflow()
for _ in range(2026):
    points.increment()

print(f"Punkte: {points.value}, Bonusleben: {points.overflow_count}")
print(isinstance(points, Counter))  # ein CounterWithOverflow ist auch ein Counter

Unterschiedliche Klassen können dieselbe Methode unterschiedlich umsetzen. Der aufrufende Code muss den genauen Typ nicht kennen (**Polymorphie**):

In [ ]:
class Shape:
    def area(self) -> float:
        raise NotImplementedError


class Rectangle(Shape):
    def __init__(self, width: float, height: float):
        self.width = width
        self.height = height

    def area(self) -> float:
        return self.width * self.height


class Circle(Shape):
    def __init__(self, radius: float):
        self.radius = radius

    def area(self) -> float:
        return 3.14159 * self.radius ** 2


for shape in [Rectangle(3, 4), Circle(1)]:
    print(f"{type(shape).__name__}: {shape.area():.2f}")

> **Vererbung sparsam einsetzen.** Sie passt, wenn gilt „X **ist ein** Y“. Oft ist es einfacher, ein Objekt als Attribut in einem anderen zu verwenden („X **hat ein** Y“).

## Übungen

**Übung 1 (Superhelden):** Schreibe eine Datenklasse `SuperHero` mit `name`, `superpower` und `strength`. Ergänze eine Methode `is_stronger_than(other)` und finde in einer Liste von Helden den stärksten.

In [ ]:
# Deine Lösung


**Übung 2 (Bankkonto):** Schreibe eine Klasse `BankAccount` mit Inhaber und Kontostand (intern als `_balance`). Methoden: `deposit(amount)`, `withdraw(amount)` und eine Property `balance`. Negative Beträge und Überziehung sollen einen `ValueError` auslösen.

In [ ]:
# Deine Lösung


**Übung 3 (Sparkonto):** Leite `SavingsAccount` von `BankAccount` ab. Es bekommt zusätzlich einen Zinssatz und eine Methode `add_interest()`, die die Zinsen gutschreibt. Nutze `super().__init__`.

In [ ]:
# Deine Lösung


**Übung 4 (Kartenspiel):** Schreibe eine Enum `Suit` mit den vier Kartenfarben (Kreuz ♣, Pik ♠, Herz ♥, Karo ♦; das Symbol als Wert) und eine Datenklasse `Card` mit `suit` und `rank`. Erzeuge mit einer List Comprehension ein Skatblatt aus 32 Karten mit den Rängen 7 bis Ass. `print(card)` soll etwa `♥ Dame` ausgeben.

In [ ]:
# Deine Lösung


## Zusammenfassung

- `class Name:` mit `__init__(self, ...)` legt Attribute an, Methoden bekommen `self`.
- `__str__` steuert die Ausgabe mit `print()`.
- `@dataclass` erzeugt `__init__`, Ausgabe und `==` automatisch.
- `_name` heißt „intern“; `@property` macht Werte lesbar, aber nicht schreibbar.
- `Enum` für feste Auswahlwerte, `StrEnum` wenn sie sich wie Strings verhalten sollen; passt zu `match`.
- Vererbung: `class Child(Parent):`, `super().methode()` ruft die Elternversion auf.